# EGM Experiment → Baihua Smoke（Notebook 版）

演示 **EGM 内部实验（XEB + Standard RB + Interleaved RB）** 数据写入 Baihua 芯片，与 **Quafu 官方校准数据** 在同一个 PostgreSQL 数据库中共存。

**架构亮点**：统一的 `analyze_task_execution_result` 根据 `task.protocol` 自动路由到对应分析器（XEB → XEB fidelity，RB → survival probability）。

**前置**：
- `db/phase1` schema + seed（010–050）已 apply
- Quafu 校准 CSV 已 ingest（`scripts/ingest/quafu_to_pg.py`）
- `psycopg[binary]` 已安装在当前 kernel 的 Python 环境中

**对应脚本**：`scripts/smoke/egm_experiment_baihua_smoke.py`

---

## §0 连接串设置

⚠️ 修改下面的 DSN 为你本地的真实值。终端 `export` 对 notebook kernel 通常 **无效**，必须在格内赋值。

In [34]:
import os, sys

os.environ["EGM_PG_DSN"] = "postgresql://ousiachai@localhost:5432/egm_phase1"

# 确保 src 在 path 中
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if os.path.isdir(os.path.join(PROJECT_ROOT, "src")):
    sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
    sys.path.insert(0, PROJECT_ROOT)

DSN = os.environ["EGM_PG_DSN"]
print(f"DSN = {DSN[:40]}..." if len(DSN) > 40 else f"DSN = {DSN}")

DSN = postgresql://ousiachai@localhost:5432/eg...


## §0.5 自检：Baihua 是否已在库中

In [35]:
from psycopg import Connection
from psycopg.rows import dict_row

with Connection.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT chip_id::text, chip_name, vendor, status FROM chip WHERE chip_name = 'Baihua'")
        chip_row = cur.fetchone()
        
        cur.execute("""
            SELECT 'qubit' AS tbl, count(*) AS cnt FROM qubit WHERE chip_id = %(cid)s::uuid
            UNION ALL
            SELECT 'coupler', count(*) FROM coupler WHERE chip_id = %(cid)s::uuid
            UNION ALL
            SELECT 'calibration_run', count(*) FROM calibration_run WHERE chip_id = %(cid)s::uuid
            UNION ALL
            SELECT 'observation_record', count(*) FROM observation_record WHERE chip_id = %(cid)s::uuid
            ORDER BY tbl
        """, {"cid": chip_row["chip_id"]})
        counts = cur.fetchall()

print(f"Chip: {chip_row['chip_name']}  vendor={chip_row['vendor']}  status={chip_row['status']}")
print(f"chip_id = {chip_row['chip_id']}")
print()
for r in counts:
    print(f"  {r['tbl']:25s} {r['cnt']:>6d} rows")

CHIP_ID = chip_row["chip_id"]

Chip: Baihua  vendor=BAQIS  status=active
chip_id = a0000001-0000-4000-8000-000000000003

  calibration_run                1 rows
  coupler                      182 rows
  observation_record           909 rows
  qubit                        156 rows


## §1 运行 EGM XEB 实验 → 写入 Baihua

使用 `DummyBackend`（`UnifiedMatrixBackend`，矩阵噪声模拟器：去极化 + T1/T2 偏置 + 相干漂移 + SPAM 误差）跑 XEB，通过 `PostgresObservationStore` 写入 `benchmark_run` + `observation_record`。

In [36]:
# pip install pydantic
import math
from datetime import datetime, timezone

from egm.analysis import analyze_task_execution_result
from egm.backends.dummy_backend_xeb import DummyBackend
from egm.datastore.postgres_observation_store import PostgresObservationStore
from egm.domain.records.task_observation_record import to_task_observation_record
from egm.execution.executor import Executor
from egm.execution.plan_runner import run_plan
from egm.schemas.configs import (
    ConfigBase, ConfigSchema, HardwareConfig, ProtocolBundle, ProtocolConfig,
)
from egm.services.planning.plan_builder import PlanBuilder
from egm.services.serialization.observation_record import to_persistence_observation_dict

from egm.protocols.physical.rb import generate_standard_rb_circuits, generate_interleaved_rb_circuits
from egm.circuits.circuit import Gate
from egm.analysis.rb import stitch_rb_results, analyze_rb_standard, calculate_epg


def _sanitize_nan(obj):
    """Recursively replace NaN/Inf with None for JSON compatibility."""
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    if isinstance(obj, dict):
        return {k: _sanitize_nan(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_sanitize_nan(v) for v in obj]
    return obj


# DummyBackend = UnifiedMatrixBackend：矩阵传播 + 物理噪声
backend = DummyBackend(cycle_fidelity=0.9996, seed=42)
print(f"Backend: {backend!r}")
print("Imports OK — unified analysis + RB/IRB ready.")

Backend: <UnifiedMatrixBackend fid=0.999600, strength=1.00, jitter=0.00050, T1=50000, T2=30000, drift=0.0020, spam=5.00e-05, cz×3.0>
Imports OK — unified analysis + RB/IRB ready.


In [37]:
def run_xeb_on_baihua(qubits, depths=(3, 5), shots=10000):
    """Run XEB on Baihua with given qubits, return observation_id."""
    config = ConfigSchema(
        base=ConfigBase(
            plan_id=f"plan-baihua-xeb-q{'_'.join(map(str, qubits))}",
            backend_name=backend.name,
        ),
        hardware=HardwareConfig(chip_name="Baihua", gate_set="native", noise_flags={}),
        protocol=ProtocolConfig(
            number_of_circuits=1,
            shots=shots,
            bundles=[ProtocolBundle(protocol="XEB", qubits=[qubits], depths=list(depths))],
        ),
    )

    plan = PlanBuilder.build_plan_from_config(config)
    executor = Executor(backend)
    exec_res = run_plan(plan, executor)

    observations = []
    for task, tr in zip(plan.tasks, exec_res.task_results):
        ar = analyze_task_execution_result(task, tr)
        observations.append(to_task_observation_record(task, tr, ar))

    dicts = [_sanitize_nan(to_persistence_observation_dict(o)) for o in observations]
    store = PostgresObservationStore(DSN)
    ids = [store.save_observation(d) for d in dicts]
    return ids[0]


print("Running XEB on qubits [0, 1] ...")
obs_id_1 = run_xeb_on_baihua([0, 1])
print(f"  observation_id = {obs_id_1}")

print("\nRunning XEB on qubits [2, 3] ...")
obs_id_2 = run_xeb_on_baihua([2, 3])
print(f"  observation_id = {obs_id_2}")

[INFO] QuantumEngine initialized with backend 'UnifiedMatrixBackend'.
[INFO] Executing 1 circuits (10000 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 1 circuits executed.
[INFO] Executing 1 circuits (10000 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 1 circuits executed.
[WARNING] [SPB-FIT] Insufficient points (<3) for SPB fit.
[WARNING] [SPB-FIT] Insufficient points (<3) for SPB fit.
[INFO] QuantumEngine initialized with backend 'UnifiedMatrixBackend'.
[INFO] Executing 1 circuits (10000 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 1 circuits executed.
[INFO] Executing 1 circuits (10000 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 1 circuits executed.
[WARNING] [SPB-FIT] Insufficient points (<3) for SPB fit.
[WARNING] [SPB-FIT] Insufficient points (<3) for SPB fit.


Running XEB on qubits [0, 1] ...
  observation_id = 5fed6e494f3d45cd97a40a690719a390

Running XEB on qubits [2, 3] ...
  observation_id = d42b40b017664c969b765fe229d7329b


## §1.5 运行 EGM Standard RB 实验 → 写入 Baihua

使用 `PlanBuilder` + 统一 `analyze_task_execution_result` 路由到 RB 分析器，计算每个 depth 的 survival probability。

In [38]:
def run_rb_on_baihua(qubits, depths=(1, 2, 5, 10), shots=2048):
    """Run Standard RB via PlanBuilder + unified analysis, save to PG."""
    config = ConfigSchema(
        base=ConfigBase(
            plan_id=f"plan-baihua-rb-q{'_'.join(map(str, qubits))}",
            backend_name=backend.name,
        ),
        hardware=HardwareConfig(chip_name="Baihua", gate_set="native", noise_flags={}),
        protocol=ProtocolConfig(
            number_of_circuits=1,
            shots=shots,
            bundles=[ProtocolBundle(protocol="RB", qubits=[qubits], depths=list(depths))],
        ),
    )

    plan = PlanBuilder.build_plan_from_config(config)
    executor = Executor(backend)
    exec_res = run_plan(plan, executor)

    observations = []
    for task, tr in zip(plan.tasks, exec_res.task_results):
        ar = analyze_task_execution_result(task, tr)
        observations.append(to_task_observation_record(task, tr, ar))

    dicts = [_sanitize_nan(to_persistence_observation_dict(o)) for o in observations]
    store = PostgresObservationStore(DSN)
    ids = [store.save_observation(d) for d in dicts]
    return ids[0]


print("Running Standard RB on qubit [0] ...")
rb_id_1 = run_rb_on_baihua([0], depths=[1, 2, 5, 10])
print(f"  observation_id = {rb_id_1}")

print("\nRunning Standard RB on qubits [0, 1] ...")
rb_id_2 = run_rb_on_baihua([0, 1], depths=[1, 2, 5])
print(f"  observation_id = {rb_id_2}")

[INFO] QuantumEngine initialized with backend 'UnifiedMatrixBackend'.
[INFO] Executing 1 circuits (2048 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 1 circuits executed.
[INFO] Executing 1 circuits (2048 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 1 circuits executed.
[INFO] Executing 1 circuits (2048 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 1 circuits executed.
[INFO] Executing 1 circuits (2048 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 1 circuits executed.
[INFO] QuantumEngine initialized with backend 'UnifiedMatrixBackend'.
[INFO] Executing 1 circuits (2048 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 1 circuits executed.
[INFO] Executing 1 circuits (2048 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 1 circuits executed.
[INFO] Executing 1 circuits (2048 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 1 circuits executed.


Running Standard RB on qubit [0] ...
  observation_id = b84e5154931a4d59b46c409d351bf143

Running Standard RB on qubits [0, 1] ...
  observation_id = b4816fb5ed2a4b4ca77fef5e766d5321


## §1.6 运行 EGM Interleaved RB (IRB) → 写入 Baihua

使用修复后的 `rb.py` 直接生成 Reference + Interleaved 线路，执行并拟合衰减曲线，计算 EPG（Error Per Gate）。

In [39]:
def run_irb_on_baihua(qubits, depths=(1, 2, 5, 10), gate_name="CZ"):
    """Run Interleaved RB: generate real Clifford circuits, fit, compute EPG."""
    interleaved_gate = Gate(gate_name, tuple(qubits[:2]))

    ref_circuits = generate_standard_rb_circuits(
        qubits=qubits, depths=list(depths), circuits_per_depth=5, seed=77
    )
    int_circuits = generate_interleaved_rb_circuits(
        qubits=qubits, depths=list(depths), circuits_per_depth=5,
        interleaved_gate=interleaved_gate, seed=77
    )

    executor = Executor(backend)
    ref_raw = executor.execute_with_ideal(ref_circuits, shots=1024)
    int_raw = executor.execute_with_ideal(int_circuits, shots=1024)

    ref_stitched = stitch_rb_results(ref_circuits, ref_raw)
    int_stitched = stitch_rb_results(int_circuits, int_raw)

    fit_ref = analyze_rb_standard(ref_stitched)
    fit_int = analyze_rb_standard(int_stitched)

    p_ref = fit_ref.alpha.value if fit_ref.success and fit_ref.alpha else 0.99
    p_int = fit_int.alpha.value if fit_int.success and fit_int.alpha else 0.98
    epg = calculate_epg(p_ref, p_int, num_qubits=len(qubits))

    print(f"  Reference EPC = {fit_ref.epc:.4e}" if fit_ref.success else "  Reference fit failed")
    print(f"  Interleaved EPC = {fit_int.epc:.4e}" if fit_int.success else "  Interleaved fit failed")
    print(f"  EPG({gate_name}) = {epg:.6f}")

    payload = _sanitize_nan({
        "protocol": "interleaved_rb",
        "chip_name": "Baihua",
        "qubits": qubits,
        "observation_time": datetime.now(timezone.utc).isoformat(),
        "execution_status": "ok",
        "analysis_status": "ok",
        "backend_name": backend.name,
        "analysis_payload": {
            "gate_name": gate_name,
            "epg": epg,
            "epc_reference": fit_ref.epc if fit_ref.success else None,
            "epc_interleaved": fit_int.epc if fit_int.success else None,
            "p_reference": p_ref,
            "p_interleaved": p_int,
            "depths": list(depths),
            "success": fit_ref.success and fit_int.success,
        },
    })

    store = PostgresObservationStore(DSN)
    return store.save_observation(payload)


print("Running Interleaved RB (CZ) on qubits [0, 1] ...")
irb_id = run_irb_on_baihua([0, 1], depths=[1, 2, 5, 10])
print(f"  observation_id = {irb_id}")

[INFO] QuantumEngine initialized with backend 'UnifiedMatrixBackend'.
[INFO] Executing 20 circuits (1024 shots each) on backend 'UnifiedMatrixBackend'.


Running Interleaved RB (CZ) on qubits [0, 1] ...


[INFO] All 20 circuits executed.
[INFO] Executing 20 circuits (1024 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 20 circuits executed.
[INFO] [RB-FIT] Success: A=0.016, p=0.94184, B=0.984, EPC=4.362e-02, R²=0.9918
[INFO] [RB-FIT] Success: A=0.182, p=0.40570, B=0.340, EPC=4.457e-01, R²=0.5323


  Reference EPC = 4.3620e-02
  Interleaved EPC = 4.4573e-01
  EPG(CZ) = 0.426938
  observation_id = 6e2fdab93b0948d3a81b73a9f052f323


## §2 验证：Baihua 的 benchmark_run 列表

In [40]:
import pandas as pd

with Connection.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT benchmark_name, status, start_time, benchmark_run_id::text
            FROM benchmark_run
            WHERE chip_id = %(cid)s::uuid
            ORDER BY start_time DESC
        """, {"cid": CHIP_ID})
        rows = cur.fetchall()

df = pd.DataFrame(rows)
print(f"{len(df)} benchmark_run(s) for Baihua:")
display(df)

# 纯文本备份
print()
print(df.to_string(index=False))

115 benchmark_run(s) for Baihua:


,benchmark_name,status,start_time,benchmark_run_id
0,interleaved_rb,succeeded,2026-05-13 15:33:39.892211+08:00,0edb10d1-0251-4227-bcf4-4abe6d82326f
1,RB,succeeded,2026-05-13 15:33:39.748320+08:00,24d29bb3-8c24-4eed-99a5-50cd09a2f886
2,RB,succeeded,2026-05-13 15:33:39.741215+08:00,b9c15c56-8dee-43e7-8050-0da0c7bd7eee
3,RB,succeeded,2026-05-13 15:33:39.734679+08:00,f2c432f4-d472-4029-bc0a-f0e0b5d5c42c
4,RB,succeeded,2026-05-13 15:33:39.725802+08:00,92c799be-bbc7-4148-8050-7677420f0af1
...,...,...,...,...
110,XEB,succeeded,2026-05-13 14:36:06.428719+08:00,24945a17-6ee0-431c-baeb-dc9cfa2feee5
111,XEB,succeeded,2026-05-13 14:25:48.802851+08:00,5312edef-d028-4dc4-a220-0db00efce600
112,XEB,succeeded,2026-05-13 14:25:48.791082+08:00,e8d37f6d-1fc5-4eda-a306-f6aa08358dfb
113,XEB,succeeded,2026-05-13 14:25:48.777073+08:00,489a5d2f-d9fc-4d4b-8cf3-aa7eb8ca0d3c



benchmark_name    status                       start_time                     benchmark_run_id
interleaved_rb succeeded 2026-05-13 15:33:39.892211+08:00 0edb10d1-0251-4227-bcf4-4abe6d82326f
            RB succeeded 2026-05-13 15:33:39.748320+08:00 24d29bb3-8c24-4eed-99a5-50cd09a2f886
            RB succeeded 2026-05-13 15:33:39.741215+08:00 b9c15c56-8dee-43e7-8050-0da0c7bd7eee
            RB succeeded 2026-05-13 15:33:39.734679+08:00 f2c432f4-d472-4029-bc0a-f0e0b5d5c42c
            RB succeeded 2026-05-13 15:33:39.725802+08:00 92c799be-bbc7-4148-8050-7677420f0af1
            RB succeeded 2026-05-13 15:33:39.719333+08:00 751e905a-892c-4120-9329-058b5d086e85
            RB succeeded 2026-05-13 15:33:39.711542+08:00 05bfb3cf-fbca-4358-8f55-c3a026ff92cc
            RB succeeded 2026-05-13 15:33:39.689977+08:00 13815bf3-93e1-490c-a84b-0f591c03b591
           XEB succeeded 2026-05-13 15:33:39.670584+08:00 156a9ba5-1e09-490e-9666-defdef7d5fef
           XEB succeeded 2026-05-13 15:33:39.6590

## §3 核心验证：两条数据流共存

按 `producer_type` 分组统计 Baihua 的 observation_record：
- `calibration_artifact` → Quafu 官方校准数据
- `benchmark_run` → EGM 内部实验（XEB / RB）

In [41]:
with Connection.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT producer_type, count(*) AS observation_count
            FROM observation_record
            WHERE chip_id = %(cid)s::uuid
            GROUP BY producer_type
            ORDER BY producer_type
        """, {"cid": CHIP_ID})
        rows = cur.fetchall()

df_sources = pd.DataFrame(rows)
display(df_sources)

print()
for _, r in df_sources.iterrows():
    label = "Quafu 官方校准" if r["producer_type"] == "calibration_artifact" else "EGM 内部实验"
    print(f"  {r['producer_type']:25s} → {r['observation_count']:>6d} observations  ({label})")

print("\n✅ 两条数据流在同一 chip_id 下共存，producer_type 完全区分。")

,producer_type,observation_count
0,calibration_artifact,806
1,benchmark_run,115



  calibration_artifact      →    806 observations  (Quafu 官方校准)
  benchmark_run             →    115 observations  (EGM 内部实验)

✅ 两条数据流在同一 chip_id 下共存，producer_type 完全区分。


## §4 抽样查看：Quafu 校准 vs EGM 实验 (q26)

对应 `sql/queries/p0/q26_calibration_vs_benchmark_compare.sql`

In [42]:
with Connection.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                o.observation_time,
                o.producer_type,
                md.metric_name,
                o.value_numeric,
                o.quality_flag,
                o.subject_type
            FROM observation_record o
            JOIN metric_definition md ON md.metric_definition_id = o.metric_definition_id
            WHERE o.chip_id = %(cid)s::uuid
              AND o.observation_time >= '2026-05-01'::timestamptz
            ORDER BY o.observation_time DESC
            LIMIT 15
        """, {"cid": CHIP_ID})
        rows = cur.fetchall()

df_compare = pd.DataFrame(rows)
print("最近 15 条 observation（校准 + 实验混合）：")
display(df_compare)

print()
print(df_compare.to_string(index=False))

最近 15 条 observation（校准 + 实验混合）：


,observation_time,producer_type,metric_name,value_numeric,quality_flag,subject_type
0,2026-05-13 15:33:39.892211+08:00,benchmark_run,egm.workflow.task_observation.v1,None,raw,chip
1,2026-05-13 15:33:39.748320+08:00,benchmark_run,egm.workflow.task_observation.v1,None,raw,chip
2,2026-05-13 15:33:39.741215+08:00,benchmark_run,egm.workflow.task_observation.v1,None,raw,chip
3,2026-05-13 15:33:39.734679+08:00,benchmark_run,egm.workflow.task_observation.v1,None,raw,chip
4,2026-05-13 15:33:39.725802+08:00,benchmark_run,egm.workflow.task_observation.v1,None,raw,chip
5,2026-05-13 15:33:39.719333+08:00,benchmark_run,egm.workflow.task_observation.v1,None,raw,chip
6,2026-05-13 15:33:39.711542+08:00,benchmark_run,egm.workflow.task_observation.v1,None,raw,chip
7,2026-05-13 15:33:39.689977+08:00,benchmark_run,egm.workflow.task_observation.v1,None,raw,chip
8,2026-05-13 15:33:39.670584+08:00,benchmark_run,egm.workflow.task_observation.v1,None,raw,chip
9,2026-05-13 15:33:39.659056+08:00,benchmark_run,egm.workflow.task_observation.v1,None,raw,chip



                observation_time producer_type                      metric_name value_numeric quality_flag subject_type
2026-05-13 15:33:39.892211+08:00 benchmark_run egm.workflow.task_observation.v1          None          raw         chip
2026-05-13 15:33:39.748320+08:00 benchmark_run egm.workflow.task_observation.v1          None          raw         chip
2026-05-13 15:33:39.741215+08:00 benchmark_run egm.workflow.task_observation.v1          None          raw         chip
2026-05-13 15:33:39.734679+08:00 benchmark_run egm.workflow.task_observation.v1          None          raw         chip
2026-05-13 15:33:39.725802+08:00 benchmark_run egm.workflow.task_observation.v1          None          raw         chip
2026-05-13 15:33:39.719333+08:00 benchmark_run egm.workflow.task_observation.v1          None          raw         chip
2026-05-13 15:33:39.711542+08:00 benchmark_run egm.workflow.task_observation.v1          None          raw         chip
2026-05-13 15:33:39.689977+08:00 benchm

## §5 抽样查看：Quafu 全芯片热力图数据 (q25)

In [43]:
with Connection.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        # 取最近一次校准的 run_id
        cur.execute("""
            SELECT calibration_run_id::text
            FROM calibration_run
            WHERE chip_id = %(cid)s::uuid
            ORDER BY start_time DESC LIMIT 1
        """, {"cid": CHIP_ID})
        run_id = cur.fetchone()["calibration_run_id"]

        cur.execute("""
            SELECT q.qubit_index, md.metric_name, o.value_numeric, o.quality_flag
            FROM observation_record o
            JOIN calibration_artifact ca
                ON ca.calibration_artifact_id = o.producer_id
                AND o.producer_type = 'calibration_artifact'
            JOIN metric_definition md ON md.metric_definition_id = o.metric_definition_id
            JOIN qubit q ON q.qubit_id = o.subject_id AND o.subject_type = 'qubit'
            WHERE ca.calibration_run_id = %(rid)s::uuid
                AND md.metric_family = 'quafu_calibration'
            ORDER BY q.qubit_index, md.metric_name
        """, {"rid": run_id})
        heatmap_rows = cur.fetchall()

df_heat = pd.DataFrame(heatmap_rows)
print(f"Calibration run: {run_id}")
print(f"Total metric observations: {len(df_heat)}")
print()

# Pivot: qubit_index × metric_name
pivot = df_heat.pivot_table(index="qubit_index", columns="metric_name", values="value_numeric")
print(f"Pivot table shape: {pivot.shape[0]} qubits × {pivot.shape[1]} metrics")
print()
print("First 10 qubits:")
display(pivot.head(10))

print()
print(pivot.head(10).to_string())

Calibration run: 71e13444-eb29-4d63-b2ec-bc6b0463477e
Total metric observations: 624

Pivot table shape: 156 qubits × 4 metrics

First 10 qubits:


metric_name,quafu_frequency,quafu_single_qubit_fidelity,quafu_t1,quafu_t2
qubit_index,,,,
0,4.468,0.999,69.450,12.319
1,4.083,0.994,86.015,7.041
2,4.402,0.998,75.448,8.280
3,3.926,0.984,58.816,24.881
4,4.457,0.999,35.625,23.724
5,4.051,0.999,26.141,10.683
6,4.389,0.997,40.629,10.087
7,4.024,0.999,38.454,5.942
8,4.460,0.980,19.076,7.638



metric_name  quafu_frequency  quafu_single_qubit_fidelity  quafu_t1  quafu_t2
qubit_index                                                                  
0                      4.468                        0.999    69.450    12.319
1                      4.083                        0.994    86.015     7.041
2                      4.402                        0.998    75.448     8.280
3                      3.926                        0.984    58.816    24.881
4                      4.457                        0.999    35.625    23.724
5                      4.051                        0.999    26.141    10.683
6                      4.389                        0.997    40.629    10.087
7                      4.024                        0.999    38.454     5.942
8                      4.460                        0.980    19.076     7.638
9                      4.161                        0.999    64.882    15.499


## §6 统计摘要

对标 Quafu 官网的 median T1 / T2 / error 等统计。

In [44]:
print("Baihua calibration statistics (from DB, validated only):")
print()

if not pivot.empty:
    for col in sorted(pivot.columns):
        valid = pivot[col].dropna()
        valid = valid[valid > 0]  # exclude suspect zeros
        if len(valid) > 0:
            print(f"  {col:35s}  median={valid.median():.4f}  min={valid.min():.4f}  max={valid.max():.4f}  count={len(valid)}")

print()
print("Compare with Quafu website:")
print("  Median T1(us):  71.608   (website)")
print("  Median T2(us):  23.724   (website)")
print("  Median 1Q err:  7.6e-4   (website)")
print("  Median 2Q err:  1.8e-2   (website)")

Baihua calibration statistics (from DB, validated only):

  quafu_frequency                      median=4.2065  min=3.7920  max=4.6110  count=156
  quafu_single_qubit_fidelity          median=0.9990  min=0.9290  max=0.9990  count=152
  quafu_t1                             median=71.6080  min=4.4410  max=177.4260  count=154
  quafu_t2                             median=23.7240  min=0.3550  max=80.6590  count=155

Compare with Quafu website:
  Median T1(us):  71.608   (website)
  Median T2(us):  23.724   (website)
  Median 1Q err:  7.6e-4   (website)
  Median 2Q err:  1.8e-2   (website)


---

**结论**：Quafu 官方校准数据（`calibration_artifact`）和 EGM 内部实验数据（`benchmark_run`）在同一个 `Baihua` chip 下共存，通过 `producer_type` 和 `source_id` 天然隔离，P0 查询均可覆盖。

对应文档：Phase 1 静态查询见 `sql/queries/p0/` 与 `docs/data-layer/query-catalog.md`

## §7 静态查询演示

展示 Phase 1 静态查询的主要使用模式。所有查询均为标准 SQL，可直接用于 API 层或 BI 工具。

**查询维度一览**：
| 维度 | 示例 |
|------|------|
| 按 protocol 筛选 | 只看 XEB / RB / IRB |
| 按 qubit/coupler 筛选 | 某个 qubit 的历史观测 |
| 按时间范围 | 今天所有实验 |
| 按数据来源 | Quafu 校准 vs EGM 实验 |
| 取 JSON 分析结果 | 从 value_json 中提取 XEB fidelity / EPG |

In [45]:
### 7.1 按 protocol 筛选 benchmark_run

# 只看某个 protocol 的实验
protocol_filter = "RB"  # 可改为 "XEB", "interleaved_rb"

with Connection.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT benchmark_name, status, start_time,
                   benchmark_run_id::text
            FROM benchmark_run
            WHERE chip_id = %(cid)s::uuid
              AND benchmark_name = %(proto)s
            ORDER BY start_time DESC
            LIMIT 10
        """, {"cid": CHIP_ID, "proto": protocol_filter})
        rows = cur.fetchall()

df_proto = pd.DataFrame(rows)
print(f"Protocol='{protocol_filter}' → {len(df_proto)} runs:")
display(df_proto)

Protocol='RB' → 10 runs:


,benchmark_name,status,start_time,benchmark_run_id
0,RB,succeeded,2026-05-13 15:33:39.748320+08:00,24d29bb3-8c24-4eed-99a5-50cd09a2f886
1,RB,succeeded,2026-05-13 15:33:39.741215+08:00,b9c15c56-8dee-43e7-8050-0da0c7bd7eee
2,RB,succeeded,2026-05-13 15:33:39.734679+08:00,f2c432f4-d472-4029-bc0a-f0e0b5d5c42c
3,RB,succeeded,2026-05-13 15:33:39.725802+08:00,92c799be-bbc7-4148-8050-7677420f0af1
4,RB,succeeded,2026-05-13 15:33:39.719333+08:00,751e905a-892c-4120-9329-058b5d086e85
5,RB,succeeded,2026-05-13 15:33:39.711542+08:00,05bfb3cf-fbca-4358-8f55-c3a026ff92cc
6,RB,succeeded,2026-05-13 15:33:39.689977+08:00,13815bf3-93e1-490c-a84b-0f591c03b591
7,RB,succeeded,2026-05-13 15:33:37.066774+08:00,44f33865-47bc-40c8-88e4-3e5f2a5ac096
8,RB,succeeded,2026-05-13 15:33:37.060116+08:00,4c0ee36d-0041-46f8-889c-181c7e1c0e07
9,RB,succeeded,2026-05-13 15:33:37.053622+08:00,396a6469-b926-4ef5-9c5d-36665c45c105


In [46]:
### 7.2 按 qubit 查询校准历史（Quafu T1 时间线）

# 查 Quafu 官方为某个 qubit 记录的 T1 历史
target_qubit_index = 0  # Baihua Q0

with Connection.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                o.observation_time,
                o.value_numeric,
                md.metric_name,
                o.quality_flag
            FROM observation_record o
            JOIN metric_definition md ON md.metric_definition_id = o.metric_definition_id
            JOIN qubit q ON q.qubit_id = o.subject_id
            WHERE o.chip_id = %(cid)s::uuid
              AND o.producer_type = 'calibration_artifact'
              AND q.qubit_index = %(qi)s
              AND md.metric_name = 'quafu_T1'
            ORDER BY o.observation_time DESC
            LIMIT 10
        """, {"cid": CHIP_ID, "qi": target_qubit_index})
        rows = cur.fetchall()

df_t1 = pd.DataFrame(rows)
print(f"Qubit Q{target_qubit_index} — T1 历史（最近 10 条）:")
display(df_t1)

Qubit Q0 — T1 历史（最近 10 条）:


""


In [47]:
### 7.3 按时间范围查所有观测（今日实验）

from datetime import date

today_str = date.today().isoformat()  # e.g. "2026-05-13"

with Connection.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                o.observation_time,
                o.producer_type,
                md.metric_name,
                o.value_numeric,
                o.subject_type
            FROM observation_record o
            LEFT JOIN metric_definition md ON md.metric_definition_id = o.metric_definition_id
            WHERE o.chip_id = %(cid)s::uuid
              AND o.observation_time::date = %(today)s::date
            ORDER BY o.observation_time DESC
            LIMIT 20
        """, {"cid": CHIP_ID, "today": today_str})
        rows = cur.fetchall()

df_today = pd.DataFrame(rows)
print(f"今日（{today_str}）观测记录（最近 20 条）:")
display(df_today)

今日（2026-05-13）观测记录（最近 20 条）:


,observation_time,producer_type,metric_name,value_numeric,subject_type
0,2026-05-13 15:33:39.892211+08:00,benchmark_run,egm.workflow.task_observation.v1,None,chip
1,2026-05-13 15:33:39.748320+08:00,benchmark_run,egm.workflow.task_observation.v1,None,chip
2,2026-05-13 15:33:39.741215+08:00,benchmark_run,egm.workflow.task_observation.v1,None,chip
3,2026-05-13 15:33:39.734679+08:00,benchmark_run,egm.workflow.task_observation.v1,None,chip
4,2026-05-13 15:33:39.725802+08:00,benchmark_run,egm.workflow.task_observation.v1,None,chip
5,2026-05-13 15:33:39.719333+08:00,benchmark_run,egm.workflow.task_observation.v1,None,chip
6,2026-05-13 15:33:39.711542+08:00,benchmark_run,egm.workflow.task_observation.v1,None,chip
7,2026-05-13 15:33:39.689977+08:00,benchmark_run,egm.workflow.task_observation.v1,None,chip
8,2026-05-13 15:33:39.670584+08:00,benchmark_run,egm.workflow.task_observation.v1,None,chip
9,2026-05-13 15:33:39.659056+08:00,benchmark_run,egm.workflow.task_observation.v1,None,chip


In [48]:
### 7.4 从 JSON 中提取分析结果（XEB fidelity / RB survival / IRB EPG）

# EGM 实验的分析结果存在 value_json → analysis_payload 中
with Connection.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                o.observation_time,
                br.benchmark_name AS protocol,
                o.value_json->'analysis_payload' AS analysis,
                o.value_json->>'protocol' AS raw_protocol
            FROM observation_record o
            JOIN benchmark_run br ON br.benchmark_run_id = o.producer_id
            WHERE o.chip_id = %(cid)s::uuid
              AND o.producer_type = 'benchmark_run'
            ORDER BY o.observation_time DESC
            LIMIT 10
        """, {"cid": CHIP_ID})
        rows = cur.fetchall()

import json as _json

print("最近 10 条 EGM 实验分析结果（从 value_json 提取）:\n")
for r in rows:
    proto = r["protocol"]
    analysis = r["analysis"]
    if analysis:
        if isinstance(analysis, str):
            analysis = _json.loads(analysis)
        if proto == "XEB":
            xeb_data = analysis.get("xeb_analysis", {})
            fid = xeb_data.get("xeb_fidelity")
            print(f"  [{proto}] fidelity={fid}")
        elif proto == "RB":
            rb_data = analysis.get("rb_analysis", {})
            surv = rb_data.get("survival_probability")
            depth = rb_data.get("depth")
            print(f"  [{proto}] depth={depth}, survival={surv}")
        elif proto == "interleaved_rb":
            epg = analysis.get("epg")
            gate = analysis.get("gate_name")
            print(f"  [{proto}] gate={gate}, EPG={epg}")
        else:
            print(f"  [{proto}] {analysis}")
    else:
        print(f"  [{proto}] (no analysis payload)")

最近 10 条 EGM 实验分析结果（从 value_json 提取）:

  [interleaved_rb] gate=CZ, EPG=0.4269384393339203
  [RB] depth=5, survival=0.99853515625
  [RB] depth=2, survival=0.9990234375
  [RB] depth=1, survival=0.99951171875
  [RB] depth=10, survival=0.9990234375
  [RB] depth=5, survival=0.99853515625
  [RB] depth=2, survival=0.99951171875
  [RB] depth=1, survival=1.0
  [XEB] fidelity=None
  [XEB] fidelity=None


In [49]:
### 7.5 Coupler CZ fidelity 时间线（Quafu 校准）

# 查看某对 coupler 的 CZ fidelity 校准历史
target_coupler_name = "CZ_0_1"  # 根据实际 coupler_name 修改

with Connection.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                o.observation_time,
                o.value_numeric AS cz_fidelity,
                o.quality_flag,
                c.coupler_name
            FROM observation_record o
            JOIN metric_definition md ON md.metric_definition_id = o.metric_definition_id
            JOIN coupler c ON c.coupler_id = o.subject_id
            WHERE o.chip_id = %(cid)s::uuid
              AND o.subject_type = 'coupler'
              AND md.metric_name = 'quafu_cz_fidelity'
              AND c.coupler_name = %(name)s
            ORDER BY o.observation_time DESC
            LIMIT 10
        """, {"cid": CHIP_ID, "name": target_coupler_name})
        rows = cur.fetchall()

df_cz = pd.DataFrame(rows)
if df_cz.empty:
    # 尝试查任意一个 coupler 作为演示
    with Connection.connect(DSN, row_factory=dict_row) as conn:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT DISTINCT c.coupler_name
                FROM observation_record o
                JOIN coupler c ON c.coupler_id = o.subject_id
                WHERE o.chip_id = %(cid)s::uuid
                  AND o.subject_type = 'coupler'
                LIMIT 5
            """, {"cid": CHIP_ID})
            available = [r["coupler_name"] for r in cur.fetchall()]
    print(f"未找到 '{target_coupler_name}'，可用 coupler names: {available}")
    if available:
        print(f"尝试查询 '{available[0]}':")
        with Connection.connect(DSN, row_factory=dict_row) as conn:
            with conn.cursor() as cur:
                cur.execute("""
                    SELECT o.observation_time, o.value_numeric AS cz_fidelity,
                           o.quality_flag, c.coupler_name
                    FROM observation_record o
                    JOIN metric_definition md ON md.metric_definition_id = o.metric_definition_id
                    JOIN coupler c ON c.coupler_id = o.subject_id
                    WHERE o.chip_id = %(cid)s::uuid
                      AND o.subject_type = 'coupler'
                      AND md.metric_name = 'quafu_cz_fidelity'
                      AND c.coupler_name = %(name)s
                    ORDER BY o.observation_time DESC
                    LIMIT 10
                """, {"cid": CHIP_ID, "name": available[0]})
                rows = cur.fetchall()
        df_cz = pd.DataFrame(rows)

print(f"CZ fidelity 时间线:")
display(df_cz)

CZ fidelity 时间线:


,observation_time,cz_fidelity,quality_flag,coupler_name
0,2026-05-13 10:40:49+08:00,0.0,suspect,CZ_0_1


In [50]:
### 7.6 跨来源对比：同一 qubit 上 Quafu 校准 vs EGM 实验

with Connection.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                o.producer_type,
                md.metric_name,
                count(*) AS records,
                avg(o.value_numeric) AS avg_value,
                min(o.observation_time) AS first_obs,
                max(o.observation_time) AS last_obs
            FROM observation_record o
            LEFT JOIN metric_definition md ON md.metric_definition_id = o.metric_definition_id
            WHERE o.chip_id = %(cid)s::uuid
            GROUP BY o.producer_type, md.metric_name
            ORDER BY o.producer_type, md.metric_name
        """, {"cid": CHIP_ID})
        rows = cur.fetchall()

df_compare = pd.DataFrame(rows)
print("全量 metric 分布（按 producer_type × metric_name）:")
display(df_compare)

全量 metric 分布（按 producer_type × metric_name）:


,producer_type,metric_name,records,avg_value,first_obs,last_obs
0,calibration_artifact,quafu_cz_fidelity,182,0.551231,2026-05-13 10:40:49+08:00,2026-05-13 10:40:49+08:00
1,calibration_artifact,quafu_frequency,156,4.212987,2026-05-13 10:40:49+08:00,2026-05-13 10:40:49+08:00
2,calibration_artifact,quafu_single_qubit_fidelity,156,0.971231,2026-05-13 10:40:49+08:00,2026-05-13 10:40:49+08:00
3,calibration_artifact,quafu_t1,156,72.679083,2026-05-13 10:40:49+08:00,2026-05-13 10:40:49+08:00
4,calibration_artifact,quafu_t2,156,26.024615,2026-05-13 10:40:49+08:00,2026-05-13 10:40:49+08:00
5,benchmark_run,egm.workflow.task_observation.v1,115,NaN,2026-05-13 14:25:48.737968+08:00,2026-05-13 15:33:39.892211+08:00


## 查询 API 小结

以上查询演示了 Phase 1 静态查询 V1 的核心能力：

| 查询模式 | SQL 关键子句 | 适用场景 |
|----------|-------------|---------|
| **按 protocol** | `WHERE benchmark_name = 'RB'` | 只看某类实验 |
| **按 qubit** | `JOIN qubit ... WHERE qubit_index = N` | 特定比特的校准历史 |
| **按时间** | `WHERE observation_time::date = ...` | 日报 / 时间窗口 |
| **按来源** | `WHERE producer_type = 'calibration_artifact'` | 区分官方 vs 内部 |
| **提取 JSON** | `value_json->'analysis_payload'->>'epg'` | 获取分析详情 |
| **跨来源聚合** | `GROUP BY producer_type, metric_name` | 数据全景 |

所有 SQL 模板见 `sql/queries/p0/q01–q26`。